# Memoria Chilena · Auditoría de metadatos RDF y enlaces documentales

**Etapa 1 · GraphRAG patrimonial — versión corregida para Jupyter**  
**Piloto:** [La religiosidad andina](https://www.memoriachilena.gob.cl/602/w3-article-790.html)

Este cuaderno **no es todavía un RAG**. Comprueba la estructura real de las fuentes antes de definir nuestra ontología o extraer entidades históricas.

**Qué hace:**
1. Consulta un minisitio y dos fichas documentales conocidas (sin recorrer el portal completo).
2. Identifica enlaces RDF, documentos vinculados y algunos campos visibles de las fichas.
3. Descarga y parsea RDF/XML cuando el servidor lo permite; conserva *exactamente* los predicados originales.
4. Exporta CSV, JSON y TTL para revisar resultados antes de construir el grafo derivado.

**No hace:** descargar libros/PDF, ejecutar OCR, entrenar modelos, ni inferir que un enlace editorial sea una relación histórica. Si hay errores HTTP/RDF, los registra sin inventar resultados.

**Respeto de fuentes:** peticiones limitadas y pausadas; revisar las condiciones de uso antes de ampliar la escala. Las investigaciones y los objetos digitales tienen regímenes de licencia diferentes. [Preguntas frecuentes](https://www.memoriachilena.gob.cl/602/w3-article-123838.html).


In [ ]:
%pip -q install requests beautifulsoup4 rdflib pandas

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
from collections import Counter
from datetime import datetime, timezone
import re, json, time, hashlib
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag
from rdflib import Graph
from IPython.display import display

BASE = "https://www.memoriachilena.gob.cl"
MINISITIO = f"{BASE}/602/w3-article-790.html"
# Dos fichas ya verificadas manualmente; NO se asume que tengan RDF válido hasta probarlo.
FICHAS = [
    f"{BASE}/602/w3-article-8648.html",  # Doctrina christiana...
    f"{BASE}/602/w3-article-8649.html",  # Chrónica del Perú...
]
PAUSA_SEGUNDOS = 2.5
SALIDA = Path("memoria_chilena_auditoria")
(SALIDA / "rdf_original").mkdir(parents=True, exist_ok=True)
SESION = requests.Session()
SESION.headers.update({"User-Agent": "Investigacion-GraphRAG-Patrimonio/0.1 (auditoria puntual)",
                       "Accept": "text/html, application/rdf+xml, application/xml;q=0.9, */*;q=0.5"})

# Se usan únicamente URLs del dominio y rutas concretas aprobadas para el piloto.
def solicitar(url, timeout=25):
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.netloc != "www.memoriachilena.gob.cl":
        raise ValueError(f"URL fuera del dominio permitido: {url}")
    time.sleep(PAUSA_SEGUNDOS)
    r = SESION.get(url, timeout=timeout, allow_redirects=True)
    r.raise_for_status()
    if urlparse(r.url).netloc != "www.memoriachilena.gob.cl":
        raise ValueError(f"Redirección fuera del dominio permitido: {r.url}")
    return r

def normalizar_espacios(s):
    return re.sub(r"\s+", " ", s or "").strip()

print("Configuración lista:", SALIDA.resolve())

## 1. Verificar HTML, enlaces RDF y documentos vinculados

La extracción de documentos usa una sección titulada «Documentos», cuando está presente. Si el sitio cambia su marcado HTML, el resultado será una lista vacía y quedará consignado; no se sustituirá con enlaces del menú general. Conservamos las dos fichas documentales verificadas como semillas explícitas.

In [ ]:
ERRORES = []
paginas = []
enlaces_documentos = []
rdf_a_descargar = []

def enlaces_de_seccion_documentos(soup, base_url):
    candidatos = []
    headings = soup.find_all(re.compile(r"^h[1-6]$"))
    inicio = next((h for h in headings if normalizar_espacios(h.get_text(" ", strip=True)).casefold() == "documentos"), None)
    if inicio is None:
        return candidatos
    vistos = set()
    for el in inicio.next_elements:
        if not isinstance(el, Tag):
            continue
        if el.name in {"h2", "h3"} and el is not inicio:
            titulo = normalizar_espacios(el.get_text(" ", strip=True)).casefold()
            if titulo in {"imágenes", "imagenes", "audiovisual", "cronología", "cronologia", "bibliografía", "bibliografia"}:
                break
        if el.name != "a":
            continue
        href = urljoin(base_url, el.get("href", ""))
        texto = normalizar_espacios(el.get_text(" ", strip=True))
        if not re.search(r"/602/w3-article-\d+\.html(?:$|[?#])", href):
            continue
        if not texto or texto.casefold() in {"descargar", "rdf", "imprimir"}:
            continue
        if href in vistos:
            continue
        vistos.add(href)
        candidatos.append({"origen": base_url, "documento_url": href, "texto_enlace": texto,
                           "evidencia": "enlace en sección Documentos del HTML"})
    return candidatos

for url in [MINISITIO] + FICHAS:
    try:
        r = solicitar(url)
        soup = BeautifulSoup(r.content, "html.parser")
        h1 = soup.find("h1")
        titulo = normalizar_espacios(h1.get_text(" ", strip=True)) if h1 else ""
        rdf_links = []
        for a in soup.find_all("a", href=True):
            texto = normalizar_espacios(a.get_text(" ", strip=True))
            href = urljoin(r.url, a["href"])
            if texto.casefold() == "rdf" and href.lower().split("?")[0].endswith(".rdf"):
                if href not in rdf_links:
                    rdf_links.append(href)
        # Si HTML contiene RDF bibliográficos múltiples, conservamos TODOS los enlaces,
        # pero para esta auditoría solo se solicita el RDF de la propia ficha (article-id).
        id_pagina = re.search(r"w3-article-(\d+)\.html", url).group(1)
        rdf_propios = [u for u in rdf_links if f"rdf-article-{id_pagina}.rdf" in u]
        pagina = {"url": url, "titulo": titulo, "http": r.status_code,
                  "rdf_total_enlaces": len(rdf_links), "rdf_propio_url": rdf_propios[0] if rdf_propios else "",
                  "n_caracteres_html": len(r.text)}
        paginas.append(pagina)
        if rdf_propios:
            rdf_a_descargar.append({"pagina_url":url, "titulo":titulo, "rdf_url":rdf_propios[0]})
        if url == MINISITIO:
            enlaces_documentos = enlaces_de_seccion_documentos(soup, url)
        print(f"OK: {titulo[:65]} | RDF propio: {len(rdf_propios)}")
    except Exception as e:
        ERRORES.append({"etapa":"html", "url":url, "error":str(e)})
        print("ERROR HTML:", url, type(e).__name__, str(e)[:150])

print("Documentos encontrados en sección:", len(enlaces_documentos))
display(pd.DataFrame(paginas))
display(pd.DataFrame(enlaces_documentos).head(12))

## 2. Descargar y analizar los RDF/XML de esas páginas

No se inventan triples. Si el endpoint falla, el error queda registrado. Conservaremos las URI originales del proveedor y las etiquetas de predicados tal como están publicadas.

In [ ]:
grafo_original = Graph()
triples_filas = []
resumen_rdf = []

for item in rdf_a_descargar:
    url = item["rdf_url"]
    try:
        r = solicitar(url)
        contenido = r.content
        id_pagina = re.search(r"rdf-article-(\d+)\.rdf", url).group(1)
        (SALIDA / "rdf_original" / f"article-{id_pagina}.rdf").write_bytes(contenido)
        parcial = Graph()
        parcial.parse(data=contenido, format="xml", publicID=url)
        grafo_original += parcial
        for s,p,o in parcial:
            triples_filas.append({"pagina_origen":item["pagina_url"],
                                 "rdf_origen":url,"sujeto":str(s),"predicado":str(p),
                                 "objeto":str(o),"objeto_tipo":type(o).__name__})
        resumen_rdf.append({"pagina_url":item["pagina_url"],"rdf_url":url,
                            "triples":len(parcial),"sujetos":len(set(parcial.subjects())),
                            "predicados":len(set(parcial.predicates())),
                            "bytes":len(contenido)})
        print(f"OK RDF {id_pagina}: {len(parcial)} triples")
    except Exception as e:
        ERRORES.append({"etapa":"rdf", "url":url, "error":str(e)})
        print("ERROR RDF:", url, type(e).__name__, str(e)[:180])

rdf_df = pd.DataFrame(triples_filas)
if len(rdf_df):
    conteo_predicados = (rdf_df.groupby("predicado").size().reset_index(name="frecuencia")
                        .sort_values("frecuencia", ascending=False))
else:
    conteo_predicados = pd.DataFrame(columns=["predicado", "frecuencia"])
print("Unión de triples RDF, sin alterar predicados:", len(grafo_original))
display(pd.DataFrame(resumen_rdf))
display(conteo_predicados.head(25))

## 3. Exportar auditoría y revisar evidencia

Se exportan las **relaciones RDF originales** por separado de los **enlaces de navegación HTML**. Los enlaces de la sección «Documentos» son asociaciones curatoriales observadas, no afirmaciones históricas sobre los contenidos de las obras.

In [ ]:
pd.DataFrame(paginas).to_csv(SALIDA/"paginas.csv", index=False)
pd.DataFrame(enlaces_documentos, columns=["origen","documento_url","texto_enlace","evidencia"]).to_csv(
    SALIDA/"enlaces_documentos.csv", index=False)
pd.DataFrame(resumen_rdf).to_csv(SALIDA/"resumen_rdf.csv", index=False)
rdf_df.to_csv(SALIDA/"triples_rdf.csv", index=False)
conteo_predicados.to_csv(SALIDA/"predicados_rdf.csv", index=False)
pd.DataFrame(ERRORES, columns=["etapa","url","error"]).to_csv(SALIDA/"errores.csv", index=False)
if len(grafo_original):
    grafo_original.serialize(destination=str(SALIDA/"rdf_original_unido.ttl"), format="turtle")

resumen={
  "fecha_utc":datetime.now(timezone.utc).isoformat(),
  "minisitio":MINISITIO,
  "paginas_html_correctas":len(paginas),
  "enlaces_documentales_en_html":len(enlaces_documentos),
  "rdf_correctos":len(resumen_rdf),
  "triples_unicos_union":len(grafo_original),
  "predicados_distintos":len(set(grafo_original.predicates())),
  "errores":len(ERRORES),
  "advertencia":"Estudio preliminar; no inferir relaciones históricas desde enlaces de navegación."
}
(SALIDA/"resumen.json").write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(resumen,indent=2,ensure_ascii=False))
import shutil
from IPython.display import FileLink, display

# En Jupyter local se genera el archivo en la carpeta del notebook.
# Solo requiere Jupyter local.
archivo_zip = shutil.make_archive(
    "memoria_chilena_resultados",
    "zip",
    root_dir=SALIDA
)
print("ZIP creado:", Path(archivo_zip).resolve())
print("Si no aparece el enlace, abre esa ruta con el explorador de archivos.")
display(FileLink(archivo_zip))


## Criterio para pasar a la etapa 2

Antes de decidir el esquema patrimonial, revisar:

- ¿Qué predicados RDF aparecen realmente? ¿Hay vínculos entre recursos o predominan propiedades literales?
- ¿Qué identificadores de documento se pueden enlazar entre las fichas HTML, RDF y PDF?
- ¿Cuántas relaciones del minisitio se recuperan en RDF y cuántas solo en HTML?
- ¿Qué campos de derechos están presentes o faltan?

**Siguiente notebook:** mapeo de ontologías (Dublin Core + PROV-O y, si hace falta, CIDOC CRM), construcción de relaciones curatoriales con procedencia y una muestra validada de documentos. Solo después: BM25 + embeddings + expansión por grafo + evaluación.

**Si el RDF no se descarga**, traer el ZIP de salida con `errores.csv`; no hay que suponer que el portal está abierto a scraping a gran escala.